# 04 — Advanced BERT Training

## Fix dari run sebelumnya
| Masalah | Fix |
|---|---|
| XLM-R masih naik di ep 8 | `epochs=12` |
| Crash → progress hilang | Checkpoint resume per fold |
| Cell 26-38 tidak jalan | Setiap cell berdiri sendiri |
| IndoBERT OOF 0.63 | `patience=4`, ep 12 |

In [ ]:
# !pip install -q torch transformers accelerate scikit-learn openpyxl

In [9]:
import sys, warnings, gc
import numpy as np, pandas as pd, torch
from pathlib import Path
warnings.filterwarnings('ignore')

# Jika di Colab: sesuaikan PROJECT_ROOT
PROJECT_ROOT = Path('/kaggle/input/datasets/ridwansigma/satriadata26')
sys.path.insert(0, str(PROJECT_ROOT))

from src import (
    Config, LABEL_ORDER, clean_text, build_label_encoder,
    run_cv, set_seed,
)
from sklearn.metrics import balanced_accuracy_score, classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [10]:
DATA_DIR   = PROJECT_ROOT / 'data'
OUTPUT_DIR = Path('/kaggle/working/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

labeled    = pd.read_excel(DATA_DIR / 'labeled/hasil_label_baru.xlsx')
to_predict = pd.read_excel(DATA_DIR / 'raw/case_1_text_to_predict.xlsx')
template   = pd.read_excel(DATA_DIR / 'raw/case_1_template_sheet.xlsx')

for df in [labeled, to_predict, template]:
    df.columns = [c.strip().lower() for c in df.columns]

labeled['label'] = labeled['label'].astype(str).str.strip()

n_dup = labeled['full_text'].duplicated().sum()
if n_dup:
    labeled = labeled.drop_duplicates(subset=['full_text']).reset_index(drop=True)
    print(f'Dropped {n_dup} duplicates')

labeled['clean']     = labeled['full_text'].astype(str).apply(clean_text)
to_predict['clean']  = to_predict['full_text'].astype(str).apply(clean_text)

le                   = build_label_encoder()
labeled['label_enc'] = le.transform(labeled['label'])
texts                = labeled['clean'].tolist()
labels               = labeled['label_enc'].values
predict_texts        = to_predict['clean'].tolist()

print(f'Train={len(texts)}  Test={len(predict_texts)}')
for i, cls in enumerate(le.classes_):
    print(f'  {cls:<22} {(labels==i).sum():>4}')

Dropped 1 duplicates
Train=4999  Test=1500
  Anggaran                767
  Distribusi              368
  Ekonomi                 196
  Kualitas Pangan        1228
  Lainnya                 570
  Politik                 780
  Sasaran Penerima        521
  Tata Kelola             569


## Exp 1 — XLM-R (ep 8 masih naik di run lama → ep 12)

**Checkpoint resume aktif** — kalau Colab crash, jalankan cell ini lagi, fold yang sudah selesai dilewati otomatis.

In [11]:
cfg_xlmr = Config(
    model_name      = 'xlm-roberta-base',
    output_dir      = str(OUTPUT_DIR),
    epochs          = 12,       # FIX: was 8
    patience        = 4,        # FIX: was 3
    llrd_factor     = 0.9,
    label_smoothing = 0.1,
    use_rdrop       = True,
    rdrop_alpha     = 0.3,
    scheduler       = 'cosine',
    lr              = 2e-5,
)
set_seed(cfg_xlmr.seed)
result_xlmr = run_cv(cfg_xlmr, texts, labels, predict_texts, label='xlmr')


=== FOLD 1/5 [xlmr] ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  fold 1 | tr=3999 val=1000 steps=1500
    ep 1/12  loss=1.9082  acc=0.4578 ★
    ep 2/12  loss=0.8521  acc=0.6616 ★
    ep 3/12  loss=0.4887  acc=0.6802 ★
    ep 4/12  loss=0.3535  acc=0.6854 ★
    ep 5/12  loss=0.2744  acc=0.6949 ★
    ep 6/12  loss=0.1969  acc=0.7011 ★
    ep 7/12  loss=0.1596  acc=0.7021 ★
    ep 8/12  loss=0.1301  acc=0.6968
    ep 9/12  loss=0.1146  acc=0.6969
    ep 10/12  loss=0.1022  acc=0.7049 ★
    ep 11/12  loss=0.0939  acc=0.7021
    ep 12/12  loss=0.0974  acc=0.7029
  fold 1 best=0.7049

=== FOLD 2/5 [xlmr] ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  fold 4 | tr=3999 val=1000 steps=1500
    ep 1/12  loss=1.8220  acc=0.4332 ★
    ep 2/12  loss=0.8815  acc=0.6348 ★
    ep 3/12  loss=0.5371  acc=0.6981 ★
    ep 4/12  loss=0.3780  acc=0.7168 ★
    ep 5/12  loss=0.2792  acc=0.7333 ★
    ep 6/12  loss=0.2034  acc=0.7200
    ep 7/12  loss=0.1630  acc=0.7407 ★
    ep 8/12  loss=0.1336  acc=0.7285
    ep 9/12  loss=0.1246  acc=0.7359
    ep 10/12  loss=0.1037  acc=0.7389
    ep 11/12  loss=0.0997  acc=0.7345
    early stop
  fold 4 best=0.7407

=== FOLD 5/5 [xlmr] ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  fold 5 | tr=4000 val=999 steps=1500
    ep 1/12  loss=1.8122  acc=0.4728 ★
    ep 2/12  loss=0.7987  acc=0.7008 ★
    ep 3/12  loss=0.4810  acc=0.7337 ★
    ep 4/12  loss=0.3590  acc=0.7379 ★
    ep 5/12  loss=0.2645  acc=0.7268
    ep 6/12  loss=0.2041  acc=0.7386 ★
    ep 7/12  loss=0.1597  acc=0.7473 ★
    ep 8/12  loss=0.1380  acc=0.7525 ★
    ep 9/12  loss=0.1083  acc=0.7593 ★
    ep 10/12  loss=0.1070  acc=0.7482
    ep 11/12  loss=0.0960  acc=0.7474
    ep 12/12  loss=0.0894  acc=0.7525
  fold 5 best=0.7593

[xlmr] folds: [np.float64(0.7049), np.float64(0.7351), np.float64(0.7139), np.float64(0.7407), np.float64(0.7593)]
[xlmr] mean ± std: 0.7308 ± 0.0194
[xlmr] OOF balanced accuracy: 0.7308
                  precision    recall  f1-score   support

        Anggaran      0.836     0.832     0.834       767
      Distribusi      0.699     0.731     0.714       368
         Ekonomi      0.730     0.760     0.745       196
 Kualitas Pangan      0.851     0.765     0.805      1228

## Exp 2 — IndoBERT-p2

In [12]:
cfg_indobert = Config(
    model_name      = 'indobenchmark/indobert-base-p2',
    output_dir      = str(OUTPUT_DIR),
    epochs          = 12,
    patience        = 4,
    llrd_factor     = 0.9,
    label_smoothing = 0.1,
    use_rdrop       = True,
    rdrop_alpha     = 0.3,
    scheduler       = 'cosine',
    lr              = 2e-5,
)
set_seed(cfg_indobert.seed)
result_indobert = run_cv(cfg_indobert, texts, labels, predict_texts, label='indobert')

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


=== FOLD 1/5 [indobert] ===


...

  fold 3 | tr=3999 val=1000 steps=1500
    ep 1/12  loss=1.6614  acc=0.4359 ★
    ep 2/12  loss=0.7956  acc=0.5699 ★
    ep 3/12  loss=0.5246  acc=0.5967 ★
    ep 4/12  loss=0.3596  acc=0.5958
    ep 5/12  loss=0.2708  acc=0.5987 ★
    ep 6/12  loss=0.2151  acc=0.6218 ★
    ep 7/12  loss=0.1781  acc=0.6234 ★
    ep 8/12  loss=0.1407  acc=0.6153
    ep 9/12  loss=0.1244  acc=0.6108
    ep 10/12  loss=0.0991  acc=0.6099
    ep 11/12  loss=0.0995  acc=0.6239 ★
    ep 12/12  loss=0.0873  acc=0.6204
  fold 3 best=0.6239

=== FOLD 4/5 [indobert] ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  fold 4 | tr=3999 val=1000 steps=1500
    ep 1/12  loss=1.6700  acc=0.4977 ★
    ep 2/12  loss=0.7920  acc=0.6402 ★
    ep 3/12  loss=0.5183  acc=0.6349
    ep 4/12  loss=0.3653  acc=0.6342
    ep 5/12  loss=0.2833  acc=0.6334
    ep 6/12  loss=0.2073  acc=0.6517 ★
    ep 7/12  loss=0.1718  acc=0.6460
    ep 8/12  loss=0.1458  acc=0.6316
    ep 9/12  loss=0.1298  acc=0.6420
    ep 10/12  loss=0.1110  acc=0.6395
    early stop
  fold 4 best=0.6517

=== FOLD 5/5 [indobert] ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  fold 5 | tr=4000 val=999 steps=1500
    ep 1/12  loss=1.6618  acc=0.4775 ★
    ep 2/12  loss=0.8143  acc=0.6073 ★
    ep 3/12  loss=0.5377  acc=0.6501 ★
    ep 4/12  loss=0.3913  acc=0.6269
    ep 5/12  loss=0.2969  acc=0.6365
    ep 6/12  loss=0.2360  acc=0.6468
    ep 7/12  loss=0.1784  acc=0.6590 ★
    ep 8/12  loss=0.1446  acc=0.6472
    ep 9/12  loss=0.1245  acc=0.6526
    ep 10/12  loss=0.1069  acc=0.6538
    ep 11/12  loss=0.1134  acc=0.6512
    early stop
  fold 5 best=0.6590

[indobert] folds: [np.float64(0.5852), np.float64(0.6183), np.float64(0.6239), np.float64(0.6517), np.float64(0.659)]
[indobert] mean ± std: 0.6276 ± 0.0263
[indobert] OOF balanced accuracy: 0.6276
                  precision    recall  f1-score   support

        Anggaran      0.750     0.764     0.757       767
      Distribusi      0.568     0.649     0.606       368
         Ekonomi      0.387     0.617     0.475       196
 Kualitas Pangan      0.820     0.623     0.708      1228
         Lainnya   

## Compare

In [ ]:
all_results = [result_xlmr, result_indobert]

print('=== MODEL COMPARISON ===')
for r in sorted(all_results, key=lambda x: x['oof_bal_acc'], reverse=True):
    mean = np.mean(r['fold_scores'])
    std  = np.std(r['fold_scores'])
    print(f'  {r["label"]:<20} mean={mean:.4f} std={std:.4f} OOF={r["oof_bal_acc"]:.4f}')

=== MODEL COMPARISON ===
  xlmr                 mean=0.7308 std=0.0194 OOF=0.7308
  indobert             mean=0.6276 std=0.0263 OOF=0.6276


## Ensemble

In [ ]:
print('Weight sweep (w_xlmr):')
best_w, best_ens = 0.5, 0.0
for w in np.arange(0.0, 1.05, 0.1):
    ens = w * result_xlmr['oof_logits'] + (1-w) * result_indobert['oof_logits']
    acc = balanced_accuracy_score(labels, ens.argmax(axis=1))
    flag = ' *' if acc > best_ens else ''
    if acc > best_ens: best_ens = acc; best_w = w
    print(f'  w={w:.1f}  bal_acc={acc:.4f}{flag}')

ens_oof  = best_w * result_xlmr['oof_logits']  + (1-best_w) * result_indobert['oof_logits']
ens_test = best_w * result_xlmr['test_logits'] + (1-best_w) * result_indobert['test_logits']
print(f'Ensemble OOF: {best_ens:.4f}  (w_xlmr={best_w:.1f})')

Weight sweep (w_xlmr):
  w=0.0  bal_acc=0.6276 *
  w=0.1  bal_acc=0.6442 *
  w=0.2  bal_acc=0.6753 *
  w=0.3  bal_acc=0.6997 *
  w=0.4  bal_acc=0.7228 *
  w=0.5  bal_acc=0.7344 *
  w=0.6  bal_acc=0.7408 *
  w=0.7  bal_acc=0.7404
  w=0.8  bal_acc=0.7360
  w=0.9  bal_acc=0.7366
  w=1.0  bal_acc=0.7308
Ensemble OOF: 0.7408  (w_xlmr=0.6)


## Per-class Recall

In [ ]:
best_single  = max(all_results, key=lambda r: r['oof_bal_acc'])
use_ensemble = best_ens > best_single['oof_bal_acc']
best_logits  = ens_oof if use_ensemble else best_single['oof_logits']
best_label   = f'ensemble_w{best_w:.1f}' if use_ensemble else best_single['label']

best_preds = best_logits.argmax(axis=1)
report = classification_report(labels, best_preds, target_names=LABEL_ORDER,
                                output_dict=True, digits=3, zero_division=0)
final_oof = max(best_ens, best_single['oof_bal_acc'])
print(f'Best: {best_label}  OOF={final_oof:.4f}')
print(f'  {"Class":<22}  Recall   F1    n')
for cls in LABEL_ORDER:
    r  = report[cls]['recall']
    f1 = report[cls]['f1-score']
    n  = int(report[cls]['support'])
    flag = '  <- LOW' if r < 0.65 else ''
    print(f'  {cls:<22}  {r:.3f}  {f1:.3f}  {n}{flag}')

Best: ensemble_w0.6  OOF=0.7408
  Class                   Recall   F1    n
  Anggaran                0.832  0.835  767
  Distribusi              0.750  0.733  368
  Ekonomi                 0.776  0.766  196
  Kualitas Pangan         0.765  0.805  1228
  Lainnya                 0.695  0.668  570
  Politik                 0.672  0.695  780
  Sasaran Penerima        0.737  0.703  521
  Tata Kelola             0.701  0.665  569


## Submit

In [ ]:
final_test = ens_test if use_ensemble else best_single['test_logits']
preds_str  = le.inverse_transform(final_test.argmax(axis=1))
sub        = template.copy(); sub['label'] = preds_str

sub_path = OUTPUT_DIR / f'submissions/submission_{best_label}.xlsx'
sub_path.parent.mkdir(exist_ok=True)
sub.to_excel(str(sub_path), index=False)
print(f'Saved : {sub_path}')
print(f'OOF   : {final_oof:.4f}')
for lbl, cnt in pd.Series(preds_str).value_counts().items():
    print(f'  {lbl:<22} {cnt:>4}  ({cnt/len(preds_str)*100:.1f}%)')

Saved : /kaggle/working/outputs/submissions/submission_ensemble_w0.6.xlsx
OOF   : 0.7408
  Kualitas Pangan         311  (20.7%)
  Anggaran                251  (16.7%)
  Politik                 204  (13.6%)
  Sasaran Penerima        174  (11.6%)
  Tata Kelola             174  (11.6%)
  Lainnya                 173  (11.5%)
  Distribusi              140  (9.3%)
  Ekonomi                  73  (4.9%)


In [ ]:
current_oof = final_oof
print(f'Current OOF: {current_oof:.4f}')
if current_oof >= 0.80:
    print('Target 80+ tercapai! Lanjut ke 05_Optuna.ipynb untuk squeeze lebih.')
else:
    gap = 0.80 - current_oof
    print(f'Gap ke 0.80: {gap:.4f}')
    if gap > 0.05: print('  -> Coba xlm-roberta-large (butuh V100/A100)')
    print('  -> Enable AWP: cfg.use_awp=True, cfg.awp_start_epoch=3')
    print('  -> Jalankan 05_Optuna.ipynb (lift 1-3 poin)')

Current OOF: 0.7408
Gap ke 0.80: 0.0592
  -> Coba xlm-roberta-large (butuh V100/A100)
  -> Enable AWP: cfg.use_awp=True, cfg.awp_start_epoch=3
  -> Jalankan 05_Optuna.ipynb (lift 1-3 poin)
